In [1]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True, cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from src.plotting import plot_autocorrelation_analysis, plot_subject_timeseries_heatmap

  DEPRECATION: Building 'src' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'src'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
import os
import numpy as np
import pandas as pd
from src.data import DataLoader

# Select the dataset to process.
# Available: 'HCP-YA', 'HCP-D', 'WAND', 'PNC', 'NKI', 'UCLA', 'Penn_LEAD', 'HBN'
dataset = 'PNC'

# Optional parcellation override. Leave as None to use the dataset-specific default
# from DATASET_PARAMS (e.g. 'Schaefer4007'). Set to e.g. 'Schaefer10007' to process
# at a different resolution. This determines the seg- token written into the scan
# registry, and therefore the parcellation the timescale outputs are saved/loaded at.
seg = None

data_loader = DataLoader(dataset=dataset)

Initialized DataLoader for dataset: PNC
  Base directory:        /mnt/storage_ssd_raid/research_data/RBC/PNC
  Raw data directory:    /mnt/storage_ssd_raid/research_data/RBC/PNC/bids
  Derivatives directory: /mnt/storage_ssd_raid/research_data/RBC/PNC/derivatives
  Output directory:      /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
  Behavioral data:       1293 subjects


# Load fMRI data

In [3]:
from src.notebook import get_dataset_params

# ── Load parameters for the selected dataset ─────────────────────────────────
# Edit DATASET_PARAMS in src/notebook.py to change processing parameters.
# Pass seg=seg to override the dataset's default parcellation (None keeps the default).
params = get_dataset_params(dataset, seg=seg)
detrend             = params['detrend']
nuisance_regression = params['nuisance_regression']
task_regression     = params['task_regression']

subject_ids = data_loader.load_subject_ids(params['subject_ids_file'])
# subject_ids = subject_ids[:10]  # limit for testing

print(f"Dataset:    {dataset}")
print(f"Modalities: {params['modalities']}")
print(f"Sessions: {params['sessions']}")
print(f"Seg: {params['seg']}")
print(f"Detrend: {detrend}")
print(f"Nuis. regression: {nuisance_regression}")
print(f"Task regression: {task_regression}")

print(f"Subjects:   {len(subject_ids)}")
subject_ids[:5]

Dataset:    PNC
Modalities: ['MRI3T']
Sessions: ['PNC1']
Seg: 4S456Parcels
Detrend: False
Nuis. regression: False
Task regression: True
Subjects:   1095


['1000393599', '1001970838', '1006151876', '1011497669', '1017092387']

In [4]:
for modality in params['modalities']:
    data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=params['sessions'],
        tasks=params['tasks'][modality],
        space=params['space'],
        seg=params['seg'],
        modality=modality,
    )

print(f"Registered {len(data_loader.scan_registry)} scans across {len(params['modalities'])} modality(ies).")

Registered 2190 scans across 1 modality(ies).


In [5]:
data_loader.validate_scan_registry(
    verbose=True,
    remove_incomplete_subjects=params['remove_incomplete_subjects'],
)

Validating 2190 scans in registry...
  Missing (file): sub-1039936647_task-rest_space-fsLR_seg-4S456Parcels_ses-PNC1_modality-MRI3T_tr-3.0_condition-rest -> /mnt/storage_ssd_raid/research_data/RBC/PNC/derivatives/xcp-d/sub-1039936647/ses-PNC1/func/sub-1039936647_ses-PNC1_task-rest_space-fsLR_seg-4S456Parcels_stat-mean_timeseries.tsv
  Missing (file): sub-1060668554_task-rest_space-fsLR_seg-4S456Parcels_ses-PNC1_modality-MRI3T_tr-3.0_condition-rest -> /mnt/storage_ssd_raid/research_data/RBC/PNC/derivatives/xcp-d/sub-1060668554/ses-PNC1/func/sub-1060668554_ses-PNC1_task-rest_space-fsLR_seg-4S456Parcels_stat-mean_timeseries.tsv
  Missing (file): sub-1060668554_task-frac2back_space-fsLR_seg-4S456Parcels_ses-PNC1_modality-MRI3T_tr-3.0_condition-task -> /mnt/storage_ssd_raid/research_data/RBC/PNC/derivatives/xcp-d/sub-1060668554/ses-PNC1/func/sub-1060668554_ses-PNC1_task-frac2back_space-fsLR_seg-4S456Parcels_stat-mean_timeseries.tsv
  Missing (file): sub-1065666069_task-rest_space-fsLR_seg-4

{'existing': [ScanInfo(subject_id='1000393599', task='rest', space='fsLR', seg='4S456Parcels', modality='MRI3T', tr=3.0, session='PNC1', stat='mean', run=None, desc=None, den=None, rbc=True),
  ScanInfo(subject_id='1000393599', task='frac2back', space='fsLR', seg='4S456Parcels', modality='MRI3T', tr=3.0, session='PNC1', stat='mean', run=None, desc=None, den=None, rbc=True),
  ScanInfo(subject_id='1001970838', task='rest', space='fsLR', seg='4S456Parcels', modality='MRI3T', tr=3.0, session='PNC1', stat='mean', run=None, desc=None, den=None, rbc=True),
  ScanInfo(subject_id='1001970838', task='frac2back', space='fsLR', seg='4S456Parcels', modality='MRI3T', tr=3.0, session='PNC1', stat='mean', run=None, desc=None, den=None, rbc=True),
  ScanInfo(subject_id='1006151876', task='rest', space='fsLR', seg='4S456Parcels', modality='MRI3T', tr=3.0, session='PNC1', stat='mean', run=None, desc=None, den=None, rbc=True),
  ScanInfo(subject_id='1006151876', task='frac2back', space='fsLR', seg='4S456

In [6]:
timeseries_data = data_loader.load_batch(
    detrend=detrend,
    nuisance_regression=nuisance_regression,
    frequency_filter=False,
    verbose=True,
)

# The XCP-D 4S456Parcels atlas returns the 400 Schaefer-400 (7Networks) cortical
# parcels followed by 56 subcortical/cerebellar parcels. Keep the cortical parcels
# only, so PNC matches the Schaefer-400 convention used by every other dataset and
# the S-A axis (verified: 4S456 columns 0:400 == schaefer400-7 order).
if params['seg'] == '4S456Parcels':
    timeseries_data = {
        scan_string: (ts[:400] if ts is not None else None)
        for scan_string, ts in timeseries_data.items()
    }

Loading 2104 scans...
  [1/2104] 1000393599 rest
  [11/2104] 1023701836 rest
  [21/2104] 1037644062 rest
  [31/2104] 1057423710 rest
  [41/2104] 1081730904 rest
  [51/2104] 1101490424 rest
  [61/2104] 1120521402 rest
  [71/2104] 1139941451 rest
  [81/2104] 1157595916 rest
  [91/2104] 1164794273 rest
  [101/2104] 1184389633 rest
  [111/2104] 1198356813 rest
  [121/2104] 121697273 rest
  [131/2104] 1228891697 rest
  [141/2104] 1254730664 rest
  [151/2104] 1299596535 rest
  [161/2104] 1310206420 rest
  [171/2104] 1337936015 rest
  [181/2104] 1357417369 rest
  [191/2104] 1391684772 rest
  [201/2104] 1412450749 rest
  [211/2104] 1440768299 rest
  [221/2104] 1452555764 rest
  [231/2104] 1467713509 rest
  [241/2104] 1483500237 rest
  [251/2104] 1494969836 rest
  [261/2104] 1512440794 rest
  [271/2104] 1521763392 rest
  [281/2104] 1545837513 rest
  [291/2104] 1569318720 rest
  [301/2104] 1585331466 rest
  [311/2104] 1592786106 rest
  [321/2104] 1605617010 rest
  [331/2104] 1616242378 rest
  [3

In [7]:
# The registry filename encodes the dataset and, for HCP-YA, the active modality.
active_modality = params['modalities'][0] if len(params['modalities']) == 1 else 'multi'
modality_suffix = {
    'MRI3T': '',
    'MRI7T': '_7T',
    'MEG':   '_MEG',
    'multi': '_multi',
}.get(active_modality, f'_{active_modality}')

# Append the parcellation only when seg was explicitly overridden, so the default
# resolution keeps the original filename (e.g. scan_registry_HCP-D.csv).
seg_suffix = f'_{seg}' if seg is not None else ''

registry_filename = f'scan_registry_{dataset}{modality_suffix}{seg_suffix}.csv'
data_loader.export_scan_registry(registry_filename)

scan_registry = data_loader.list_registered_scans()
scan_registry.head()

,sub,ses,task,space,seg,modality,tr,condition,run,desc,den,rbc,stat,scan_string
0,1000393599,PNC1,rest,fsLR,4S456Parcels,MRI3T,3.0,rest,None,None,None,True,mean,sub-1000393599_task-rest_space-fsLR_seg-4S456P...
1,1000393599,PNC1,frac2back,fsLR,4S456Parcels,MRI3T,3.0,task,None,None,None,True,mean,sub-1000393599_task-frac2back_space-fsLR_seg-4...
2,1001970838,PNC1,rest,fsLR,4S456Parcels,MRI3T,3.0,rest,None,None,None,True,mean,sub-1001970838_task-rest_space-fsLR_seg-4S456P...
3,1001970838,PNC1,frac2back,fsLR,4S456Parcels,MRI3T,3.0,task,None,None,None,True,mean,sub-1001970838_task-frac2back_space-fsLR_seg-4...
4,1006151876,PNC1,rest,fsLR,4S456Parcels,MRI3T,3.0,rest,None,None,None,True,mean,sub-1006151876_task-rest_space-fsLR_seg-4S456P...


In [8]:
subject_ids = scan_registry['sub'].unique()
n_subs = len(subject_ids)
print(n_subs)

1052


In [9]:
subject_ids

array(['1000393599', '1001970838', '1006151876', ..., '985910486',
       '987544292', '997818717'], shape=(1052,), dtype=object)

# Process fMRI timescales

In [10]:
run_cell = True

In [11]:
if run_cell:
    timescales_processor = Timescales(dataset=dataset, timeseries_data=timeseries_data, verbose=True)
    if task_regression:
        timescales_processor.regress_task_structure()

Initialized DataLoader for dataset: PNC
  Base directory:        /mnt/storage_ssd_raid/research_data/RBC/PNC
  Raw data directory:    /mnt/storage_ssd_raid/research_data/RBC/PNC/bids
  Derivatives directory: /mnt/storage_ssd_raid/research_data/RBC/PNC/derivatives
  Output directory:      /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
  Behavioral data:       1293 subjects

	Initialized Timescales for dataset: PNC
	Number of scans: 2104
	Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales


Regressing task structure out of timeseries: 100%|██████████| 2104/2104 [00:20<00:00, 102.90it/s]


### Save out time series

for others 

In [12]:
if run_cell:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'timeseries', dataset)
    os.makedirs(out_dir, exist_ok=True)

    seg = params['seg']
    tasks = sorted(set(
        part[len('task-'):]
        for k in timeseries_data
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_ts = {k: v for k, v in timeseries_data.items() if f'task-{task}_' in k}
        if not task_ts:
            continue
        out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}_data-ts.npy')
        np.save(out_path, task_ts, allow_pickle=True)
        print(f"  {task}: {len(task_ts)} scans  →  {out_path}")

        if task_regression:
            task_resid = {k: v for k, v in timescales_processor.timeseries_residuals.items()
                          if f'task-{task}_' in k and v is not None}
            if task_resid:
                out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}_data-tsresid.npy')
                np.save(out_path, task_resid, allow_pickle=True)
                print(f"  {task}: {len(task_resid)} residuals  →  {out_path}")

  frac2back: 1052 scans  →  /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/timeseries/PNC/PNC_4S456Parcels_task-frac2back_data-ts.npy
  frac2back: 1052 residuals  →  /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/timeseries/PNC/PNC_4S456Parcels_task-frac2back_data-tsresid.npy
  rest: 1052 scans  →  /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/timeseries/PNC/PNC_4S456Parcels_task-rest_data-ts.npy


In [13]:
if run_cell and task_regression:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'task_contrasts', dataset)
    os.makedirs(out_dir, exist_ok=True)

    seg = params['seg']
    tasks = sorted(set(
        part[len('task-'):]
        for k in timescales_processor.task_contrasts
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_data = {k: v for k, v in timescales_processor.task_contrasts.items()
                     if f'task-{task}_' in k and v is not None}
        if not task_data:
            continue
        out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}_data-task_contrasts.npy')
        np.save(out_path, task_data, allow_pickle=True)
        print(f"  {task}: {len(task_data)} scans  →  {out_path}")

  frac2back: 1052 scans  →  /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/task_contrasts/PNC/PNC_4S456Parcels_task-frac2back_data-task_contrasts.npy


In [14]:
# timescales_processor.get_min_scan_length_per_subject()

### Timeseries metrics (not timescales)

In [15]:
if run_cell:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, force_recompute=False, verbose=True)

        if task_regression:
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, force_recompute=False, verbose=True)

Computing metric_tsnr: 100%|██████████| 2104/2104 [00:01<00:00, 1729.28it/s]



metric_tsnr computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing metric_tsnr_resids: 100%|██████████| 2104/2104 [00:01<00:00, 1639.43it/s]



metric_tsnr_resids computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing metric_var: 100%|██████████| 2104/2104 [00:01<00:00, 1765.60it/s]



metric_var computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing metric_var_resids: 100%|██████████| 2104/2104 [00:01<00:00, 1727.92it/s]


metric_var_resids computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


### Timescale measures

In [16]:
timescale_measures = [
    'auc_zero_crossing',
    'zero_crossing',
    # 'zero_crossing_robust_nc2',
    # 'zero_crossing_robust_nc3',
    # 'auc_half_crossing',
    # 'decay_time_half',
    # 'lag_one',

    # 'tau',
    # 'tau_howell',
    # 'tau_murray_par',
    # 'B_murray_par',
    # 'B_murray_par_lag3mins',
    # 'B_murray_par_lag4mins',
    # 'B_murray_par_lag5mins',
]

In [17]:
if run_cell:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, force_recompute=False, verbose=True)
        
    autocorr_dict_raw = timescales_processor.autocorr_dict.copy()
    timescales_dict_raw = timescales_processor.timescales_dict.copy()

Computing autocorr: 100%|██████████| 2104/2104 [00:30<00:00, 69.06it/s]



autocorr computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing auc_zero_crossing timescales: 100%|██████████| 2104/2104 [00:03<00:00, 637.47it/s]



auc_zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing zero_crossing timescales: 100%|██████████| 2104/2104 [00:01<00:00, 1075.50it/s]


zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


In [18]:
if run_cell and task_regression:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='resids', force_recompute=False, verbose=True)
        
    autocorr_dict_resid = timescales_processor.autocorr_dict.copy()
    timescales_dict_resid = timescales_processor.timescales_dict.copy()

Computing autocorr_resids: 100%|██████████| 2104/2104 [00:29<00:00, 70.56it/s]



autocorr_resids computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing auc_zero_crossing timescales: 100%|██████████| 2104/2104 [00:03<00:00, 639.41it/s]



auc_zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


Computing zero_crossing timescales: 100%|██████████| 2104/2104 [00:01<00:00, 1085.55it/s]


zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed:    2104
  Total:             2104


### Frequency specific timescales

In [19]:
run_cell = True

In [20]:
if run_cell and (dataset == 'HCP-YA' or dataset == 'HCP-D'):
    filter_configs = [
        {'lowpass': 0.08, 'highpass': 0.01},  # bandpass (standard)
        # {'lowpass': 0.15, 'highpass': 0.08},  # bandpass (higher freqs)
        # # just the ultra-low freqs
        # {'lowpass': 0.01, 'highpass': None},  # lowpass
        # # filter out the ultra-low freqs
        # {'lowpass': None, 'highpass': 0.01},  # highpass (standard)
        
        # {'lowpass': 0.08, 'highpass': None},  # lowpass (standard)
        # # increasing high-freq in bandpass
        # {'lowpass': 0.1, 'highpass': 0.01},  # bandpass
        # {'lowpass': 0.2, 'highpass': 0.01},  # bandpass

        # # just the high freqs
        # {'lowpass': None, 'highpass': 0.08},  # lowpass
        
        # limited bandpass
        # {'lowpass': 0.04, 'highpass': 0.01},  # bandpass
        # {'lowpass': 0.03, 'highpass': 0.01},  # bandpass
        # {'lowpass': 0.02, 'highpass': 0.01},  # bandpass

        # moving bandpass
        # {'lowpass': 0.07, 'highpass': None},  # bandpass
        # {'lowpass': 0.08, 'highpass': 0.01},  # bandpass
        # {'lowpass': 0.09, 'highpass': 0.02},  # bandpass
        # {'lowpass': 0.1, 'highpass': 0.03},  # bandpass
        # {'lowpass': 0.11, 'highpass': 0.04},  # bandpass
        # {'lowpass': 0.12, 'highpass': 0.05},  # bandpass
        # {'lowpass': 0.13, 'highpass': 0.06},  # bandpass
        # {'lowpass': 0.14, 'highpass': 0.07},  # bandpass
    ]

    for cfg in filter_configs:
        print(cfg)
        lowpass  = cfg['lowpass']
        highpass = cfg['highpass']

        if highpass is not None and lowpass is not None:
            suffix = f'bp-{highpass}-{lowpass}hz'
        elif highpass is not None:
            suffix = f'hp-{highpass}hz'
        elif lowpass is not None:
            suffix = f'lp-{lowpass}hz'

        # Load with filtering applied before task regression
        timeseries_data_filt = data_loader.load_batch(
            detrend=detrend,
            nuisance_regression=nuisance_regression,
            frequency_filter=True,
            lowpass=lowpass,
            highpass=highpass,
            verbose=True,
        )

        processor_filt = Timescales(dataset=dataset, timeseries_data=timeseries_data_filt, verbose=False)
        if task_regression:
            processor_filt.regress_task_structure()

        # lowpass/highpass passed for output naming only; filtering already applied at load time
        processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False,
                                                        lowpass=lowpass, highpass=highpass, verbose=False)
        for timescale_measure in timescale_measures:
            processor_filt.compute_timescale_measures(method=timescale_measure, use_residuals=False,
                                                      suffix=suffix, force_recompute=False, verbose=False)

        if task_regression:
            processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False,
                                                            lowpass=lowpass, highpass=highpass, verbose=False)
            for timescale_measure in timescale_measures:
                processor_filt.compute_timescale_measures(method=timescale_measure, use_residuals=True,
                                                          suffix=f'{suffix}_resids', force_recompute=False, verbose=False)
                
        # save out for Emily
        timeseries_data_out = dict()
        for scan_key in processor_filt.timeseries_data.keys():
            if 'task-rfMRIREST' in scan_key:
                timeseries_data_out[scan_key] = np.array(processor_filt.timeseries_data[scan_key]).copy()
        print(len(list(timeseries_data_out.keys())))
        np.save(os.path.expanduser(f'~/Desktop/emily_share/HCP-YA_Schaefer400-7_task-rfMRIREST_filter-{highpass}-{lowpass}_data-ts.npy'), timeseries_data_out)

        tasks = ['task-tfMRIWM', 'task-tfMRIEMOTION', 'task-tfMRIGAMBLING', 'task-tfMRILANGUAGE', 'task-tfMRIRELATIONAL', 'task-tfMRISOCIAL']
        for task in tasks:
            timeseries_data_out = dict()
            for scan_key in processor_filt.timeseries_data.keys():
                if task in scan_key:
                    timeseries_data_out[scan_key] = np.array(processor_filt.timeseries_data[scan_key]).copy()
            print(len(list(timeseries_data_out.keys())))      
            np.save(os.path.expanduser(f'~/Desktop/emily_share/HCP-YA_Schaefer400-7_{task}_filter-{highpass}-{lowpass}_data-ts.npy'), timeseries_data_out)

            timeseries_data_out = dict()
            for scan_key in processor_filt.timeseries_residuals.keys():
                if task in scan_key:
                    timeseries_data_out[scan_key] = np.array(processor_filt.timeseries_residuals[scan_key]).copy()
            print(len(list(timeseries_data_out.keys())))  
            np.save(os.path.expanduser(f'~/Desktop/emily_share/HCP-YA_Schaefer400-7_{task}_filter-{highpass}-{lowpass}_data-tsresid.npy'), timeseries_data_out)